# Total Return Swap Pricing — Lou (2018)

**Reference:** W. Lou, *Pricing Total Return Swap*, 2018. SSRN 3217420.

Visual validation of equity TRS with repo financing, trinomial tree, and XVA decomposition.

## Contents
1. Default dashboard (auto 2x2)
2. Value vs spot — pre-crisis vs repo-adjusted (Eq 2 vs 7)
3. FVA vs repo spread (Eq 8)
4. Tree convergence to analytic (Section 8.4)
5. XVA decomposition (Section 5)
6. ATI funding spread validation

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

import math
from datetime import date, timedelta
from pricebook.discount_curve import DiscountCurve
from pricebook.trs_lou import TotalReturnSwapLou
from pricebook.viz import plot, PlotBuilder

REF = date(2026, 4, 26)
r = 0.10; T = 1.0
curve = DiscountCurve.flat(REF, r)
D = math.exp(-r * T)
libor = (1/D - 1) / T

trs = TotalReturnSwapLou(
    spot=100.0, funding_rate=libor + 0.02,
    repo_spread=0.05, maturity=T, sigma=0.20)

print(f"S0={trs.spot}, r={r}, D={D:.4f}, Libor(sc)={libor:.4f}")
print(f"Funding rate={trs.funding_rate:.4f}, repo spread={trs.repo_spread*10000:.0f}bp")

## 1. Default Dashboard

Auto-detected 2x2: value vs spot, FVA vs repo, tree convergence, summary table.

In [ ]:
plot(trs, curve)

## 2. Value vs Spot — Pre-Crisis vs Repo-Adjusted

Pre-crisis (Eq 2) vs full-CSA with repo (Eq 7). The repo-adjusted value is lower by the FVA.

In [ ]:
PlotBuilder(trs, curve).payoff().figure()

## 3. FVA vs Repo Spread

$\text{FVA} = (e^{(r_s - r)T} - 1) \times S_t \geq 0$

The hedge financing cost grows exponentially with the repo-OIS spread.

In [ ]:
PlotBuilder(trs, curve).greeks().figure()

## 4. Tree Convergence to Analytic

The trinomial tree (Eq 12) with full CSA should converge to the analytic closed form (Eq 7) as n_steps increases.

In [ ]:
PlotBuilder(trs, curve).comparison().figure()

## 5. XVA Decomposition

$U = V^* - V = \text{CVA} - \text{DVA} + \text{CFA} - \text{DFA}$

With asymmetric funding ($r_b \neq r_c$), the switching discount rate creates an XVA wedge.

In [ ]:
PlotBuilder(trs, curve).payoff().comparison().summary().figure()

## 6. ATI Validation

At-the-issue: with $s_f = 0$ and ATI Libor $\ell = (1/D - 1)/T$, $V_0 = 0$ exactly.

In [ ]:
# ATI: sf = 0, rf = libor => V = 0 exactly
from pricebook.trs_lou import trs_equity_full_csa

trs_ati = TotalReturnSwapLou(
    spot=100.0, funding_rate=libor,
    repo_spread=0.0, maturity=T, sigma=0.20)
result_ati = trs_equity_full_csa(trs_ati.spot, trs_ati.spot, libor, T, 0.0, D, rs_minus_r=0.0)
print(f"V at ATI (sf=0, rs=r): {result_ati.value:.2e}  (should be ~0)")

PlotBuilder(trs_ati, curve).payoff().summary().figure()